# Amortization ablation — four routes to the same Q* (Heston world, C3)

SMT is one way to reach the projected measure. This is the comparison the paper promised and never
showed. Design and predictions are pre-registered in `taskc/DECISIONS.md` section 24, written before
this ran.

| arm | what it is | needs target samples? |
|---|---|---|
| **SMT** | frozen P_θ + frozen h_ψ, corrected sampler | no |
| **(a) Q-trained** | DDPM trained on Heston-Q paths, frozen recipe | **yes** (oracle) |
| **(b) L\*-resampled** | DDPM trained on P_θ paths resampled by L\*, frozen recipe | no |
| **SIR** | self-normalised weights on a fresh P_θ draw, resampled | no |

All four are evaluated in the **same z coordinates**: arm (a) reuses the frozen standardizer rather than
refitting one, so nothing is compared across different scalings.

In [ ]:
PINNED_COMMIT    = "__PINNED__"
NOTEBOOK_VERSION = "ablate-2026.09.25a"
EXPECT_TASKC     = "taskc-2026.09.25a"

%cd /content
%rm -rf ddpm_option_pricing
!git clone -q https://github.com/nilay47/ddpm_option_pricing.git
%cd ddpm_option_pricing
!git fetch -q --all
!git checkout -q {PINNED_COMMIT}
import subprocess
HEAD = subprocess.run(["git","rev-parse","HEAD"], capture_output=True, text=True).stdout.strip()
print("pinned :", PINNED_COMMIT); print("HEAD   :", HEAD); print("notebook:", NOTEBOOK_VERSION)
assert HEAD.startswith(PINNED_COMMIT) or PINNED_COMMIT.startswith(HEAD), "checkout missed the pinned commit -- stop"
!git log --oneline -1

### Stage 0 — environment, full fp32 (TF32 off), Drive

In [ ]:
import os, sys, json, math, time
import torch
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision("highest")

if os.path.basename(os.getcwd()) == "notebooks": os.chdir("..")
sys.path.insert(0, os.getcwd())
import numpy as np
from dataclasses import replace
import taskc
from taskc.config import CFG, FROZEN, frozen
from taskc.data import PathStandardizer, make_loader
from taskc.ptheta import (make_schedule, build_model, train_ptheta_decay_ema, save_checkpoint,
                          load_checkpoint, sample_ptheta)
from taskc.hnet import HNet, load_hnet
from taskc.smt import sample_smt, sliced_wasserstein, resample_weighted
from taskc.dual import solve_level
from taskc.gate import all_columns
from config import (P_PARAMS, q_params, SimConfig, SIM_P, CALIB_TESTFUNS, VANILLA_C3,
                    HELDOUT_TESTFUNS, HELDOUT_VANILLAS, EXOTICS)
from heston import simulate
from constraints import build
import evaluation as ev

_nb_path = "notebooks/taskc_09_ablation_colab.ipynb"
# INVARIANT: PINNED_COMMIT must name a commit whose copy of this notebook already
# carries NOTEBOOK_VERSION, so a version bump takes two commits -- one to land the
# version, one to point the pin at it.
assert taskc.__version__ == EXPECT_TASKC, f"stale: taskc {taskc.__version__} != {EXPECT_TASKC}"
assert NOTEBOOK_VERSION in open(_nb_path).read(), (
    f"{NOTEBOOK_VERSION} does not appear in the copy of this notebook at PINNED_COMMIT "
    f"({PINNED_COMMIT[:7]}). Either this is a stale cached notebook -- reopen it from the "
    f"pinned URL -- or the pin was not advanced together with the version.")

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
try:
    from google.colab import drive; drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/ddpm_ablation"
except Exception:
    DRIVE = os.environ.get("DRY_OUT", "artifacts_taskc/ablation_local")
os.makedirs(DRIVE, exist_ok=True)
print("taskc:", taskc.__version__, "| torch:", torch.__version__, "| device:", DEVICE,
      "| tf32:", torch.backends.cuda.matmul.allow_tf32,
      "| matmul:", torch.get_float32_matmul_precision())
print("out ->", DRIVE)

### Stage 0b — the frozen models

`artifacts_taskc/` is gitignored, so the frozen P_θ and h_ψ behind the paper's primary amortization
result are **not in the repo**. Upload them once to `<DRIVE>/frozen/`:

```
ptheta_mlp21.pt   <- artifacts_taskc/lrema/ptheta_mlp21.pt
hpsi_C3.pt        <- artifacts_taskc/lrema/hpsi/hpsi_C3.pt
```

The cell below fails with that message if they are missing, rather than silently training new ones.

In [ ]:
FROZEN_DIR = os.path.join(DRIVE, "frozen")
os.makedirs(FROZEN_DIR, exist_ok=True)
P_CKPT = os.path.join(FROZEN_DIR, "ptheta_mlp21.pt")
H_CKPT = os.path.join(FROZEN_DIR, "hpsi_C3.pt")
for pth, src in ((P_CKPT, "artifacts_taskc/lrema/ptheta_mlp21.pt"),
                 (H_CKPT, "artifacts_taskc/lrema/hpsi/hpsi_C3.pt")):
    assert os.path.exists(pth), (
        f"missing {pth}. Upload it from {src} -- these are the frozen models behind the "
        f"paper's primary amortization result and must NOT be retrained here.")

N_POOL   = 1_000_000     # fresh P_theta draw: dual solve + SIR pool + Q* reference
N_EVAL   = 100_000       # paths per arm for the comparison
N_TRAIN  = 100_000       # training paths for arms (a) and (b), matching P_theta's own
N_TRUTH  = 4_000_000     # Heston-Q Monte Carlo truth for the exotics
LEVEL    = "C3"
SKIP_DONE = True

RUN = frozen(artifact_dir=DRIVE)
RES = os.path.join(DRIVE, "ablation.json")
res = json.load(open(RES)) if (SKIP_DONE and os.path.exists(RES)) else dict(
    notebook=NOTEBOOK_VERSION, pinned=PINNED_COMMIT, taskc=taskc.__version__, device=DEVICE,
    device_name=(torch.cuda.get_device_name(0) if DEVICE=="cuda" else DEVICE),
    fp32=dict(tf32=torch.backends.cuda.matmul.allow_tf32, precision=torch.get_float32_matmul_precision()),
    level=LEVEL, n_pool=N_POOL, n_eval=N_EVAL, n_train=N_TRAIN, n_truth=N_TRUTH,
    stages={}, timings={})
def save(): json.dump(res, open(RES,"w"), indent=1, default=float)
save()

model, std, ck_cfg, ck_extra = load_checkpoint(P_CKPT, device=DEVICE)
assert ck_cfg["epochs"] == 450 and ck_cfg["lr_decay"] and ck_cfg["ema"], f"not the frozen recipe: {ck_extra}"
hnet = load_hnet(H_CKPT, device=DEVICE)
sched = make_schedule(RUN, device=DEVICE)
print("frozen P_theta:", ck_extra)
print("frozen standardizer:", std)
print("stages done:", list(res["stages"]))

### Stage 1 — the target: a fresh P_θ pool, the dual solve, and Q\* as a weighted sample

The dual is solved on the fresh pool, giving L\* and the self-normalised weights. Q\* is realised as an
SIR resample of that pool, which is the reference the sliced-Wasserstein distances are measured against.

In [ ]:
POOL = os.path.join(DRIVE, "pool_z.npz")
if "pool" not in res["stages"]:
    t0 = time.time()
    d = sample_ptheta(model, sched, n=N_POOL, seed=RUN.solve_draw.seed, cfg=RUN,
                      device=DEVICE, verbose=True)
    np.savez_compressed(POOL, z=d.z.astype(np.float32))
    res["stages"]["pool"] = dict(n=int(d.z.shape[0]), rejected=int(d.n_rejected), seconds=time.time()-t0)
    res["timings"]["pool"] = time.time()-t0; save()
z_pool = np.load(POOL)["z"].astype(np.float64)
pool_paths = std.to_paths(z_pool, world="Ptheta_pool")
q = q_params()
tilt, rsolve, cs_pool = solve_level(LEVEL, pool_paths, q)
w_pool = rsolve.w
ess = float(1.0/np.sum(w_pool**2)/len(w_pool))
print(f"pool {z_pool.shape}  dual: m={cs_pool.m}  ESS {ess*100:.2f}%  KL {rsolve.kl:.4f}")
z_qstar = resample_weighted(z_pool, w_pool, N_EVAL, seed=4001)     # Q* as samples
res["stages"]["dual"] = dict(m=int(cs_pool.m), ess=ess, kl=float(rsolve.kl),
                             unique_qstar=float(len(np.unique(z_qstar, axis=0))/N_EVAL))
save()
print(f"Q* reference sample: {z_qstar.shape}, unique rows "
      f"{res['stages']['dual']['unique_qstar']*100:.1f}%")

### Stage 2 — arm (a): a DDPM trained on Heston-Q paths

The oracle. It is the only arm that sees samples from the target law, and is labelled as such
everywhere it appears.

In [ ]:
A_CKPT = os.path.join(DRIVE, "ptheta_qtrained.pt")
if "train_a" not in res["stages"]:
    t0 = time.time()
    simQ = SimConfig(n_paths=N_TRAIN, H=RUN.H, dt=RUN.dt, seed=SIM_P.seed + 900_000)
    Yq = simulate(q, simQ, world="Q").returns()
    zq = std.to_z(Yq)                       # FROZEN standardizer, not a refit
    keep = np.abs(zq).max(axis=1) <= RUN.z_cap
    zq = zq[keep].astype(np.float32)
    print(f"arm (a) training set {zq.shape}, {int((~keep).sum())} rejected by the cap")
    ma = build_model(RUN).to(DEVICE)
    la = make_loader(zq, batch_size=RUN.batch_size, seed=RUN.init_seed)
    ma = train_ptheta_decay_ema(ma, la, sched, RUN, device=DEVICE); ma.eval()
    save_checkpoint(A_CKPT, ma, std, RUN, extra=dict(arm="a_qtrained", train_seconds=time.time()-t0))
    res["stages"]["train_a"] = dict(n=int(zq.shape[0]), rejected=int((~keep).sum()),
                                    seconds=time.time()-t0)
    res["timings"]["train_a"] = time.time()-t0; save()
    print(f"arm (a) trained in {(time.time()-t0)/60:.1f} min")
model_a, _, _, _ = load_checkpoint(A_CKPT, device=DEVICE)

### Stage 3 — arm (b): a DDPM trained on P_θ paths resampled by L\*

The retrain alternative. Its training set is an SIR resample of the pool by L\*, so it targets the same
Q\* as SMT but reaches it by fitting rather than by correcting.

In [ ]:
B_CKPT = os.path.join(DRIVE, "ptheta_lstar.pt")
if "train_b" not in res["stages"]:
    t0 = time.time()
    zb = resample_weighted(z_pool, w_pool, N_TRAIN, seed=4002).astype(np.float32)
    uniq = float(len(np.unique(zb, axis=0))/len(zb))
    print(f"arm (b) training set {zb.shape}, unique rows {uniq*100:.1f}%")
    mb = build_model(RUN).to(DEVICE)
    lb = make_loader(zb, batch_size=RUN.batch_size, seed=RUN.init_seed)
    mb = train_ptheta_decay_ema(mb, lb, sched, RUN, device=DEVICE); mb.eval()
    save_checkpoint(B_CKPT, mb, std, RUN, extra=dict(arm="b_lstar", train_seconds=time.time()-t0))
    res["stages"]["train_b"] = dict(n=int(zb.shape[0]), unique_frac=uniq, seconds=time.time()-t0)
    res["timings"]["train_b"] = time.time()-t0; save()
    print(f"arm (b) trained in {(time.time()-t0)/60:.1f} min")
model_b, _, _, _ = load_checkpoint(B_CKPT, device=DEVICE)

### Stage 4 — draw 10⁵ paths from each arm, timed

SIR draws from a *fresh* pool held out from the one the dual was solved on, so its resampling error is
not measured on the sample that produced its own weights.

In [ ]:
DRAWS = os.path.join(DRIVE, "arm_draws.npz")
if "draws" not in res["stages"]:
    t0 = time.time(); tim = {}
    t = time.time(); d_smt = sample_smt(model, hnet, sched, N_EVAL, 5001, RUN, DEVICE, verbose=False)
    tim["SMT"] = time.time()-t
    t = time.time(); d_a = sample_ptheta(model_a, sched, n=N_EVAL, seed=5002, cfg=RUN, device=DEVICE, verbose=False)
    tim["a_qtrained"] = time.time()-t
    t = time.time(); d_b = sample_ptheta(model_b, sched, n=N_EVAL, seed=5003, cfg=RUN, device=DEVICE, verbose=False)
    tim["b_lstar"] = time.time()-t
    t = time.time()
    d_fresh = sample_ptheta(model, sched, n=N_POOL, seed=5004, cfg=RUN, device=DEVICE, verbose=False)
    fresh_paths = std.to_paths(d_fresh.z.astype(np.float64), world="Ptheta_fresh")
    cs_f = build(fresh_paths, CALIB_TESTFUNS, VANILLA_C3, q)
    w_f = tilt.weights(cs_f.G)
    z_sir = resample_weighted(d_fresh.z.astype(np.float64), w_f, N_EVAL, seed=5005)
    tim["SIR"] = time.time()-t
    np.savez_compressed(DRAWS, smt=d_smt.z.astype(np.float32), a=d_a.z.astype(np.float32),
                        b=d_b.z.astype(np.float32), sir=z_sir.astype(np.float32),
                        qstar=z_qstar.astype(np.float32))
    res["stages"]["draws"] = dict(sample_seconds=tim,
                                  sir_pool_ess=float(1/np.sum(w_f**2)/len(w_f)),
                                  sir_unique=float(len(np.unique(z_sir, axis=0))/N_EVAL),
                                  smt_rejected=int(d_smt.n_rejected))
    res["timings"]["draws"] = time.time()-t0; save()
    for k, v in tim.items(): print(f"  {k:12s} {v:7.1f}s for {N_EVAL:,} paths")
D = np.load(DRAWS)
ARMS = {"SMT": D["smt"], "a_qtrained": D["a"], "b_lstar": D["b"], "SIR": D["sir"]}
print({k: v.shape for k, v in ARMS.items()})

### Stage 5 — metrics

**Columns.** Every one of the 77 columns has an exact target under Heston-Q: the martingale columns are
zero under any Q, the vanillas are Carr-Madan prices. Error is (mean − target) / SE.

**Exotics** against a large-N Heston-Q Monte Carlo. **Sliced Wasserstein** against the Q\* resample.

In [ ]:
if "truth" not in res["stages"]:
    t0 = time.time()
    pq = simulate(q, SimConfig(n_paths=N_TRUTH, H=RUN.H, dt=RUN.dt, seed=90210), world="Q")
    tr = {k: dict(price=float(ev.exotic_payoff(pq, **sp).mean()),
                  se=float(ev.exotic_payoff(pq, **sp).std(ddof=1)/math.sqrt(pq.n)))
          for k, sp in EXOTICS.items()}
    res["stages"]["truth"] = tr; res["timings"]["truth"] = time.time()-t0; save()
    for k, v in tr.items(): print(f"  {k:28s} {v['price']:9.5f} +- {v['se']:.5f}")
TRUTH = res["stages"]["truth"]

# exact column targets under Heston-Q, in the order all_columns() returns
_p0 = std.to_paths(ARMS["SMT"][:100].astype(np.float64))
_cs = build(_p0, CALIB_TESTFUNS, VANILLA_C3, q)
_ho = build(_p0, HELDOUT_TESTFUNS, HELDOUT_VANILLAS, q)
TARGETS = np.concatenate([_cs.c, _ho.c])

if "metrics" not in res["stages"]:
    t0 = time.time(); out = {}
    for arm, z in ARMS.items():
        p = std.to_paths(z.astype(np.float64), world=arm)
        G, names, kinds, calibrated = all_columns(p)
        mean = G.mean(0); se = G.std(0)/math.sqrt(G.shape[0])
        se = np.where(se > 0, se, np.inf)
        zsc = (mean - TARGETS)/se
        ex = {}
        for k, sp in EXOTICS.items():
            pay = ev.exotic_payoff(p, **sp)
            m = float(pay.mean()); s = float(pay.std(ddof=1)/math.sqrt(len(pay)))
            tp, ts_ = TRUTH[k]["price"], TRUTH[k]["se"]
            ex[k] = dict(price=m, se=s, err=m-tp, err_se=(m-tp)/math.sqrt(s**2+ts_**2),
                         err_pct=100*(m-tp)/tp)
        sw = float(sliced_wasserstein(z.astype(np.float64), D["qstar"].astype(np.float64),
                                      n_proj=256, seed=7))
        out[arm] = dict(
            calib=dict(max_abs_z=float(np.abs(zsc[calibrated]).max()),
                       rms_z=float(np.sqrt((zsc[calibrated]**2).mean())),
                       worst=str(np.array(names)[calibrated][int(np.argmax(np.abs(zsc[calibrated])))])),
            heldout=dict(max_abs_z=float(np.abs(zsc[~calibrated]).max()),
                         rms_z=float(np.sqrt((zsc[~calibrated]**2).mean())),
                         worst=str(np.array(names)[~calibrated][int(np.argmax(np.abs(zsc[~calibrated])))])),
            exotics=ex, sliced_wasserstein=sw)
        print(f"{arm:12s} calib |z| max {out[arm]['calib']['max_abs_z']:6.2f} rms {out[arm]['calib']['rms_z']:5.2f} | "
              f"held-out max {out[arm]['heldout']['max_abs_z']:6.2f} rms {out[arm]['heldout']['rms_z']:5.2f} | "
              f"SW {sw:.5f}", flush=True)
    res["stages"]["metrics"] = out; res["timings"]["metrics"] = time.time()-t0; save()
print("metrics done")

### Stage 6 — the table

In [ ]:
M = res["stages"]["metrics"]; S = res["stages"]["draws"]["sample_seconds"]
LAB = {"SMT":"SMT (frozen h_psi)","a_qtrained":"(a) Q-trained  [ORACLE: uses target samples]",
       "b_lstar":"(b) L*-resampled retrain","SIR":"SIR on fresh P_theta"}
TRAIN_S = {"SMT": 0.0, "a_qtrained": res["stages"]["train_a"]["seconds"],
           "b_lstar": res["stages"]["train_b"]["seconds"], "SIR": 0.0}
print(f"{'arm':44s} {'calib |z|':>12s} {'held |z|':>12s} {'SW':>9s} {'train s':>9s} {'1e5 s':>8s}")
print("-"*100)
for a in ARMS:
    print(f"{LAB[a]:44s} {M[a]['calib']['max_abs_z']:6.2f}/{M[a]['calib']['rms_z']:5.2f} "
          f"{M[a]['heldout']['max_abs_z']:6.2f}/{M[a]['heldout']['rms_z']:5.2f} "
          f"{M[a]['sliced_wasserstein']:9.5f} {TRAIN_S[a]:9.0f} {S[a]:8.1f}")
print("\n(max/rms of |mean - exact Heston-Q target| / SE, over 53 calibrated and 24 held-out columns)")
print(f"\n{'exotic':28s} " + " ".join(f"{LAB[a].split(' ')[0]:>18s}" for a in ARMS) + f" {'Heston-Q':>11s}")
print("-"*110)
for k in EXOTICS:
    print(f"{k:28s} " + " ".join(f"{M[a]['exotics'][k]['price']:9.5f}({M[a]['exotics'][k]['err_se']:+6.2f})"
                                 for a in ARMS) + f" {TRUTH[k]['price']:11.5f}")
print("\nvalues are price(error in SE against Heston-Q truth)")
print(f"\nSIR pool ESS {res['stages']['draws']['sir_pool_ess']*100:.2f}%  "
      f"unique draws {res['stages']['draws']['sir_unique']*100:.1f}%  "
      f"| Q* reference unique {res['stages']['dual']['unique_qstar']*100:.1f}%")
print("timings (min):", {k: round(v/60,2) for k,v in res["timings"].items()})

Results in `ablation.json` on Drive. Predictions in DECISIONS.md section 24.3 are checked against
this table and any that fail are recorded there as falsified.